# DAP - Hate Speech Classification with LLMs

This notebook explores using Large Language Models (LLMs) for hate speech classification on the HateXplain dataset.
We will use a prompt engineering approach to classify text into "hatespeech", "normal", or "offensive". Select T4 GPU!

In [6]:
# @title Setup Ollama and Dependencies
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
48 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as re

In [85]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [8]:
!ollama pull gemma3:4b

In [9]:
!pip install langchain-ollama

In [10]:
# @title Test Inference using Sample Prompt

from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
from IPython.display import Markdown

template = """Question: {question}

Answer: Let's think step by step."""

prompt = ChatPromptTemplate.from_template(template)

model = OllamaLLM(model="gemma3:4b")

chain = prompt | model

display(Markdown(chain.invoke({"question": "What's the length of hypotenuse in a right angled triangle"})))

Okay, that’s a great start to a problem-solving approach!  “Let’s think step by step” indicates you’re going to break down the problem. 

Now, you need to actually *tell* me how to find the length of the hypotenuse.  

Here's the breakdown of how to approach this:

1. **Recall the Pythagorean Theorem:** The Pythagorean Theorem states that in a right-angled triangle, the square of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the other two sides (the legs).  It's written as:  a² + b² = c²
   *  `a` and `b` are the lengths of the two shorter sides (legs) of the triangle.
   *  `c` is the length of the hypotenuse (the side opposite the right angle).

2. **Identify the knowns:** You need to know the lengths of the two legs of the right-angled triangle.

3. **Solve for the hypotenuse:**  Rearrange the equation to solve for `c`:
   *  c² = a² + b²
   *  c = √(a² + b²)  (Take the square root of both sides)

**Now, can you finish your response by telling me how to calculate the length of the hypotenuse given the lengths of the two legs?**  For example, you could say: "To find the length of the hypotenuse (c), you square the lengths of the two legs (a² and b²), add them together, and then take the square root of the result."

In [89]:
# @title Pull new model (e.g., phi3)
# If you want to use a different model, first pull it.
# For example, to pull 'phi3':
!ollama pull phi3

In [90]:
# @title Update the LLM model instance
# Change the model name here to the one you pulled (e.g., 'phi3')
from langchain_ollama.llms import OllamaLLM

model = OllamaLLM(model="phi3")

print(f"Model updated to: {model.model}")

# You would then need to re-run the classification and evaluation cells
# (e.g., 'Run Inference on Test Set' and 'Model Evaluation Metrics')
# to see the performance of the new model.

Model updated to: phi3


In [11]:
# @title Install Packages to Load Dataset

!pip install -q OpenAI "datasets<3.0" "pyarrow<20" scikit-learn pandas python-dotenv tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 16.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [30]:
import os
import json
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Initialize OpenAI Client for Local Ollama
# Ollama provides an OpenAI-compatible API at /v1
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama" # API key is required but ignored by Ollama
)

print("Client initialized for local Ollama.")

Client initialized for local Ollama.


In [13]:
# @title Data Processing Helpers (Reused from BERT Notebook)

ID2NAME = {0: "hatespeech", 1: "normal", 2: "offensive"}
NAME2ID = {v:k for k,v in ID2NAME.items()}

def _coerce_annotators(ann):
    if ann is None: return []
    if isinstance(ann, str):
        try: ann = json.loads(ann)
        except: return []
    if isinstance(ann, list):
        if len(ann) == 0: return []
        if isinstance(ann[0], dict): return ann
        out = []
        for x in ann:
            try: out.append({"label": int(x), "target": []})
            except: pass
        return out
    if isinstance(ann, dict):
        labels = ann.get("label", [])
        targets = ann.get("target", [])
        n = len(labels)
        out = []
        for i in range(n):
            try: lab = int(labels[i])
            except: continue
            tgt = targets[i] if (isinstance(targets, list) and i < len(targets)) else []
            out.append({"label": lab, "target": tgt if isinstance(tgt, list) else [tgt]})
        return out
    return []

def _coerce_rationales(rats):
    if rats is None: return []
    if isinstance(rats, str):
        try: rats = json.loads(rats)
        except: return []
    if isinstance(rats, list):
        out = []
        for r in rats:
            if isinstance(r, list):
                out.append([int(v) if isinstance(v, (int, bool)) else 0 for v in r])
        return out
    return []

def _collect_targets(anns):
    seen, out = set(), []
    for a in anns:
        for t in a.get("target", []) or []:
            if t not in seen:
                seen.add(t); out.append(t)
    return out

def _majority_vote_label(anns):
    pri = [0, 2, 1]
    counts = {0:0, 1:0, 2:0}
    for a in anns:
        lab = int(a.get("label", 1))
        if lab in counts: counts[lab] += 1
    if counts[0] == counts[1] == counts[2] == 0:
        return 1
    return sorted(counts.items(), key=lambda kv: (-kv[1], pri.index(kv[0])))[0][0]

def _majority_rationale(rats, n_tokens):
    if not rats: return [0] * n_tokens
    thresh = (len(rats) + 1) // 2
    agg = [0] * n_tokens
    for r in rats:
        for i, v in enumerate(r[:n_tokens]):
            agg[i] += int(v)
    return [1 if c >= thresh else 0 for c in agg]

def _pick_text(ex):
    if "post_tokens" in ex and isinstance(ex["post_tokens"], list):
        return " ".join(ex["post_tokens"])
    if "text" in ex: return str(ex["text"])
    if "cleaned_text" in ex: return str(ex["cleaned_text"])
    for v in ex.values():
        if isinstance(v, list) and all(isinstance(t, str) for t in v):
            return " ".join(v)
    return ""

def normalize_example(e):
    tokens = e.get("post_tokens", None)
    text = _pick_text(e)
    anns = _coerce_annotators(e.get("annotators", None))
    rats = _coerce_rationales(e.get("rationales", None))
    label = _majority_vote_label(anns)
    rat_mask = _majority_rationale(rats, len(tokens) if isinstance(tokens, list) else len(text.split()))
    targets = _collect_targets(anns)
    return {
        "text_norm": text.strip(),
        "label_norm": int(label),
        "rationale_mask": rat_mask,
        "targets": targets,
    }

In [14]:
# Load and Prepare Data
print("Loading HateXplain dataset...")
ds = load_dataset("Hate-speech-CNERG/hatexplain", trust_remote_code=True)

# Normalize
# Removed num_proc=2 to avoid pickling errors on Windows where helper functions aren't found in worker processes
ds = ds.map(normalize_example)

# Keep only necessary columns
to_keep = ["text_norm", "label_norm"]
drop_cols = [c for c in ds["train"].column_names if c not in to_keep]
ds = ds.remove_columns(drop_cols)

# Filter empty
ds = ds.filter(lambda t: len(t.strip()) > 0, input_columns=["text_norm"])

Loading HateXplain dataset...


Generating train split:   0%|          | 0/15383 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1922 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1924 [00:00<?, ? examples/s]

Map:   0%|          | 0/15383 [00:00<?, ? examples/s]

Map:   0%|          | 0/1922 [00:00<?, ? examples/s]

Map:   0%|          | 0/1924 [00:00<?, ? examples/s]

Filter:   0%|          | 0/15383 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1922 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1924 [00:00<?, ? examples/s]

In [83]:
# @title Select Test Set

import numpy as np

# Select Test Set
test_ds = ds["test"]
print(f"Full Test set size: {len(test_ds)}")
# Select Test Set
test_ds = ds["test"].select(range(100))
print(f"Selected Test set size: {len(test_ds)}")

Full Test set size: 1924
Selected Test set size: 100


In [97]:
# @title Define Prompt & Run on 1 Example

import json
import re

def parse_llm_response(response_text):
    # Default values
    label = "normal"
    reasoning = "N/A - Could not parse reasoning from LLM response."

    # 1. Try straightforward JSON parsing
    try:
        data = json.loads(response_text)
        label = data.get("label", "normal").lower()
        reasoning = data.get("reasoning", "N/A - Reasoning not found in JSON.")
        return {"label": label, "reasoning": reasoning}
    except json.JSONDecodeError:
        pass

    # 2. Fallback: Use Regex to find the JSON object if the model chatted extra text
    try:
        match = re.search(r'\{.*?\}', response_text, re.DOTALL)
        if match:
            json_str = match.group(0)
            data = json.loads(json_str)
            label = data.get("label", "normal").lower()
            reasoning = data.get("reasoning", "N/A - Reasoning not found in JSON (regex fallback).")
            return {"label": label, "reasoning": reasoning}
    except:
        pass

    # 3. Emergency Fallback: Look for the label explicitly in text
    text_lower = response_text.lower()
    if "hatespeech" in text_lower: label = "hatespeech"
    elif "offensive" in text_lower: label = "offensive"
    else: label = "normal"

    return {"label": label, "reasoning": reasoning} # Keep default reasoning if JSON not found

# Define the system prompt globally for easy access and export
system_prompt = (
    "You are a strict text classifier for research. You categorize text into: 'hatespeech', 'offensive', or 'normal'.\n"
    "Respond ONLY with a valid JSON object: {\"reasoning\": \"...\", \"label\": \"...\"}\n\n"

    "### DEFINITIONS\n"
    "1. HATESPEECH (Identity + Dehumanization/Violence):\n"
    "   - Target: Must be a protected group (race, religion, gender, sexuality, etc.).\n"
    "   - Attack: Must be severe: dehumanization (vermin/filth), calls for violence/death, claims of biological inferiority, or racial slurs used to attack.\n"
    "   - NOT Hatespeech: Political criticism, 'I hate [Group]', or rude insults without dehumanization.\n\n"

    "2. OFFENSIVE (Rude/Vulgar):\n"
    "   - Profanity, sexual vulgarity, or personal insults (idiot, ugly).\n"
    "   - Hostility toward non-protected groups (landlords, police, politicians).\n"
    "   - General rude statements like 'Men are trash' or 'F*** [Group]' (unless accompanied by dehumanization).\n\n"

    "3. NORMAL (Safe/Context):\n"
    "   - Neutral statements, factual discussion, or disagreement.\n"
    "   - Criticism of ideas/doctrine (e.g., 'Catholicism is wrong' is Normal).\n"
    "   - Reporting on hate speech (quoting slurs).\n\n"

    "### FEW-SHOT EXAMPLES (Use these as ground truth)\n"
    "Input: 'The government needs to stop letting these migrants in, they are ruining the economy.'\n"
    "Output: {\"reasoning\": \"Hostile toward a group (migrants) and discusses exclusion (stop letting in), but frames it as policy/economy. No slurs or dehumanization.\", \"label\": \"normal\"}\n\n"

    "Input: 'Migrants are filthy cockroaches that need to be exterminated.'\n"
    "Output: {\"reasoning\": \"Explicitly targets protected group (migrants) with dehumanization (cockroaches) and calls for violence (exterminated).\", \"label\": \"hatespeech\"}\n\n"

    "Input: 'You are a f***ing idiot and I hope you fail.'\n"
    "Output: {\"reasoning\": \"Contains profanity and hostility, but targets an individual with generic insults. No identity attack.\", \"label\": \"offensive\"}\n\n"

    "Input: 'I absolutely hate Jews.'\n"
    "Output: {\"reasoning\": \"Expression of hatred toward a protected group, but lacks specific dehumanization, slurs, or calls for violence. It is borderline, but strictly fits Offensive/Hatespeech boundary depending on codebook. Defaulting to lower severity if no specific threat exists.\", \"label\": \"offensive\"}\n"
    "*(Note: Adjust this last example based on strictly how you want to handle 'I hate X' statements. Most strict classifiers label pure 'I hate X' as Hatespeech, but your rules imply needing a 'severe' signal like vermin/inferiority. If 'I hate X' is hate speech for you, change the label to 'hatespeech'.)*\n\n"

    "### FINAL INSTRUCTION\n"
    "Prioritize the reasoning step to check for Hard Gates. When in doubt between Offensive/Hatespeech, choose Offensive."
)

def classify_text(text, model="gemma3:4b"):
    """
    Classifies text using Local Ollama API.
    Make sure you have pulled the model: `ollama pull phi3`
    """
    global system_prompt # Reference the global system_prompt

    user_prompt = f"Text: {text}"

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0, # Deterministic
            max_tokens=100, # Increased max_tokens to allow for full JSON response
            timeout=60
        )
        return parse_llm_response(response.choices[0].message.content.strip())
    except Exception as e:
        print(f"Error: {e}")
        return {"label": "error", "reasoning": str(e)}

print(f"Full Entry: {test_ds[1]}")
sample_text, sample_true_label = test_ds[42]['text_norm'], ID2NAME[test_ds[42]['label_norm']] # change both the indexes to match
# label norm = 0 hatespeech , 1 normal , 2 offensive
print(f"Sample Text: {sample_text}")
print(f"Ground Truth: {sample_true_label}")
prediction_output = classify_text(sample_text)
print(f"Prediction: {prediction_output['label']}")
print(f"Reasoning: {prediction_output['reasoning']}")

Full Entry: {'text_norm': '<user> you are missing an essential premise whether any of those immigrants committed any crimes dumbass', 'label_norm': 1}
Sample Text: <user> <user> <user> <user> <user> <user> yeah just the backwoods redneck portion of society in oklahoma that does not even know other sports exist
Ground Truth: normal
Prediction: hatespeech
Reasoning: Targets a group (Oklahoma residents) with derogatory terms ('backwoods', 'redneck') implying inferiority and exclusion. While not explicitly violent, it utilizes dehumanizing stereotypes and exhibits hostility toward a specific geographic group. This meets the criteria for Hatespeech due to the use of derogatory labels and implied hostility.


In [93]:
# @title Run Inference on Test Set
predictions = []
ground_truth = []
texts = []
reasonings = [] # New list to store reasoning from the model

print("Running inference...")
for example in tqdm(test_ds):
    text = example['text_norm']
    label_id = example['label_norm']
    true_label = ID2NAME[label_id]

    prediction_output = classify_text(text)
    pred_label = prediction_output['label']
    reasoning = prediction_output['reasoning']

    # Ensure the predicted label is one of the valid categories
    # The parse_llm_response function should handle most cases, but this is a final safeguard
    if pred_label not in NAME2ID:
        pred_label = "normal" # Default fallback for truly unexpected output

    predictions.append(pred_label)
    ground_truth.append(true_label)
    texts.append(text)
    reasonings.append(reasoning)

# Create a DataFrame for results, including the reasoning
results_df = pd.DataFrame({
    "text": texts,
    "ground_truth": ground_truth,
    "prediction": predictions,
    "reasoning": reasonings
})

results_df.head()

Running inference...


100%|██████████| 100/100 [03:32<00:00,  2.13s/it]


,text,ground_truth,prediction,reasoning
0,<user> men can not be raped can not be abused that why they call it violence against women and children because men are always the perpetrators women come first even before children,normal,hatespeech,N/A - Could not parse reasoning from LLM response.
1,<user> you are missing an essential premise whether any of those immigrants committed any crimes dumbass,normal,offensive,"Targets an individual ('dumbass') with a rude insult. While critical of immigrants, it lacks any identity attack, dehumanization, or calls for violence. It does not meet the criteria for Hatespeech. It is an offensive statement due to the insult."
2,<user> <user> why are you repeating yourself are you a little retarded,offensive,offensive,"Targets an individual with a derogatory slur ('retarded') and a personal insult ('you are a little'). While it's offensive, it lacks the severe dehumanization or calls for violence necessary for 'hatespeech' classification. It primarily constitutes a personal attack."
3,<user> <user> well she ’ muslim so of course she hates the jews jew hatred is one of the defining feature of islam,hatespeech,hatespeech,"This statement targets a protected group (Muslims) and makes a generalization about their beliefs (hatred of Jews) as a defining feature of Islam. This implies a negative stereotype and prejudice based on religious identity, which constitutes hate speech. The statement relies on a harmful generalization and promotes prejudice."
4,<user> lol not me i don ’ t deal with bms not outside children ’ ss i leave that ghetto mess for everybody else 😂 😂 😂,offensive,normal,N/A - Could not parse reasoning from LLM response.


In [88]:
# @title Display Full Results
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

print("Complete Results - Full Text View:\n")
print(results_df.to_string(index=True))

Complete Results - Full Text View:

                                                                                                                                                                                                                                                                                                       text ground_truth  prediction                                                                                                                                                                                                                                                                                                                                                                                                                                                                 reasoning
0                                                                                                                     <user> men can not be raped can not be abused that why they cal

In [94]:
# @title Model Evaluation Metrics
from IPython.display import Markdown

print("\nClassification Report:")
report = classification_report(ground_truth, predictions, target_names=["hatespeech", "normal", "offensive"], labels=["hatespeech", "normal", "offensive"], output_dict=True)
display(pd.DataFrame(report).transpose().round(2))

print("\nConfusion Matrix:")
cm = confusion_matrix(ground_truth, predictions, labels=["hatespeech", "normal", "offensive"])
cm_df = pd.DataFrame(cm, index=["True Hatespeech", "True Normal", "True Offensive"], columns=["Pred Hatespeech", "Pred Normal", "Pred Offensive"])
display(cm_df)


Classification Report:


,precision,recall,f1-score,support
hatespeech,0.47,0.87,0.61,31.00
normal,0.73,0.25,0.37,44.00
offensive,0.37,0.40,0.38,25.00
accuracy,0.48,0.48,0.48,0.48
macro avg,0.52,0.51,0.45,100.00
weighted avg,0.56,0.48,0.45,100.00



Confusion Matrix:


,Pred Hatespeech,Pred Normal,Pred Offensive
True Hatespeech,27,1,3
True Normal,19,11,14
True Offensive,12,3,10


classification metrics:

Low recall for a class = it missed many true items of that class (many false negatives).

Low precision for a class = many of its predictions for that class were wrong (many false positives).


In [100]:
# @title Export Results and Prompt to Google Sheet
from datetime import datetime
from google.colab import auth
import gspread
from google.auth import default
# from google.colab import sheets # Removed as we're using gspread for direct Drive save
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# Authenticate Google Colab to access Google Drive/Sheets
auth.authenticate_user()

# Authorize gspread
creds, _ = default()
gc = gspread.authorize(creds)

# Generate a timestamp for the spreadsheet title
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
spreadsheet_title = f"HateSpeech_LLM_Results_{timestamp}"

print(f"Creating and exporting results to a new Google Sheet in your Drive: {spreadsheet_title}")

try:
    # Create a new spreadsheet in Google Drive
    spreadsheet = gc.create(spreadsheet_title)

    # Print the URL for easy access
    print(f"Spreadsheet created at: {spreadsheet.url}")

    # --- Export results_df to a new worksheet ---
    # Add a new worksheet titled 'Results' or rename the default 'Sheet1'
    try:
        worksheet_results = spreadsheet.worksheet('Sheet1')
        worksheet_results.update_title('Results')
    except gspread.exceptions.WorksheetNotFound:
        worksheet_results = spreadsheet.add_worksheet(title="Results", rows="100", cols="20")

    # Convert DataFrame to a list of lists (including header) for gspread
    results_data = [results_df.columns.values.tolist()] + results_df.values.tolist()
    worksheet_results.update(range_name='A1', values=results_data)

    # --- Export system_prompt to a new worksheet ---
    worksheet_prompt = spreadsheet.add_worksheet(title="System_Prompt", rows="1", cols="1")
    prompt_data = [[system_prompt]] # gspread expects a list of lists
    worksheet_prompt.update(range_name='A1', values=prompt_data)

    # --- Export Classification Report to a new worksheet ---
    # Recalculate report for export (as it's often local to a cell)
    report = classification_report(ground_truth, predictions, target_names=["hatespeech", "normal", "offensive"], labels=["hatespeech", "normal", "offensive"], output_dict=True)
    report_df = pd.DataFrame(report).transpose().round(2)
    # Include the index as a column for export
    worksheet_report = spreadsheet.add_worksheet(title="Classification_Report", rows=len(report_df)+1, cols=len(report_df.columns) + 1)
    worksheet_report.update(range_name='A1', values=[report_df.reset_index().columns.values.tolist()] + report_df.reset_index().values.tolist())

    # --- Export Confusion Matrix to a new worksheet ---
    # Recalculate cm_df for export (as it's often local to a cell)
    cm = confusion_matrix(ground_truth, predictions, labels=["hatespeech", "normal", "offensive"])
    cm_df = pd.DataFrame(cm, index=["True Hatespeech", "True Normal", "True Offensive"], columns=["Pred Hatespeech", "Pred Normal", "Pred Offensive"])
    # Include the index as a column for export
    worksheet_cm = spreadsheet.add_worksheet(title="Confusion_Matrix", rows=len(cm_df)+1, cols=len(cm_df.columns) + 1)
    worksheet_cm.update(range_name='A1', values=[cm_df.reset_index().columns.values.tolist()] + cm_df.reset_index().values.tolist())

    print("Export complete! You can find the spreadsheet in your Google Drive under 'My Drive'.")

except Exception as e:
    print(f"Error exporting to Google Sheets: {e}")

Creating and exporting results to a new Google Sheet in your Drive: HateSpeech_LLM_Results_20251212_124248
Spreadsheet created at: https://docs.google.com/spreadsheets/d/1bXNRsmTZhza0C4u9OptcP-SmYIveVnBRLseiMUTpBac
Export complete! You can find the spreadsheet in your Google Drive under 'My Drive'.


In [ ]:
# @title Old Prompt Storage (Gemma 3)

## V1
def classify_text(text, model="gemma3:4b"):
    """
    Classifies text using Local Ollama API.
    Make sure you have pulled the model: `ollama pull phi3`
    """
    system_prompt = (
        "You are a precise and objective content moderation AI. Your task is to classify text into one of three categories: 'hatespeech', 'offensive', or 'normal'.\n\n"
        "You must prioritize distinct differentiation between 'rude/vulgar' content and actual 'hate speech'. Do not trigger on keywords alone; analyze the intent and target.\n\n"

        "### 1. CATEGORY: hatespeech\n"
        "**Definition:** Content that promotes violence, incites hatred, promotes discrimination, or disparages on the basis of PROTECTED GROUPS.\n"
        "**Protected Groups:** Race, ethnicity, religion, disability, age, nationality, veteran status, sexual orientation, gender, or gender identity.\n"
        "**Key Criteria (Must meet at least one):**\n"
        "- **Dehumanization:** Comparing the group to animals, insects, filth, or bacteria.\n"
        "- **Violence:** Calls for death, assault, or exclusion of the group.\n"
        "- **Inferiority:** Statements implying the group is mentally, morally, or physically inferior based on their identity.\n"
        "- **Slurs:** Use of racial/ethnic slurs intended to degrade.\n\n"

        "### 2. CATEGORY: offensive\n"
        "**Definition:** Content that is rude, crude, disrespectful, or vulgar, but DOES NOT meet the criteria for hate speech.\n"
        "**Key Criteria:**\n"
        "- **Profanity:** Strong swearing or obscenity (e.g., 'This is f**king stupid').\n"
        "- **Personal Insults:** attacking an individual’s intelligence, appearance, or character (e.g., 'You are an idiot,' 'You look ugly') without referencing a protected group.\n"
        "- **General Vulgarity:** Sexual or scatological humor that is in poor taste.\n"
        "**Crucial Distinction:** If someone is simply being mean, angry, or vulgar, but not attacking a protected identity, it is 'offensive', not 'hatespeech'.\n\n"

        "### 3. CATEGORY: normal\n"
        "**Definition:** Content that is neutral, factual, positive, or purely conversational. Includes constructive criticism, disagreement, and non-vulgar negative opinions (e.g., 'I really disliked that movie').\n\n"

        "### 4. FALSE POSITIVE PREVENTION (Read Carefully)\n"
        "To avoid being over-sensitive, you must adhere to these exceptions:\n"
        "- **Criticism of Ideas vs. People:** Criticizing a religion, ideology, or government is NOT hate speech unless it dehumanizes the individuals belonging to that group.\n"
        "- **Reclaimed Speech:** Users self-identifying with a term (e.g., 'I am a queer man') is Normal/Safe.\n"
        "- **Counter-Speech:** Quoting hate speech to condemn it or educate others is Normal/Safe.\n"
        "- **Factual Discussion:** Discussing sensitive topics (crime statistics, biology, immigration) neutrally without slurs or calls for violence is Normal/Safe.\n\n"

        "### Few-shot Examples:\n"
        "Text: <user> <user> well she ’ muslim so of course she hates the jews jew hatred is one of the defining feature of islam\n"
        "Answer: hatespeech\n\n"
        "Text: <user> you are missing an essential premise whether any of those immigrants committed any crimes dumbass\n"
        "Answer: offensive\n\n"
        "Text: apple watch series <number> available in titanium black and white and ceramic new nike models as well apple event apple watch apple\n"
        "Answer: normal\n\n"

        "### OUTPUT FORMAT\n"
        "Provide ONLY the category name as your response, in lowercase: 'hatespeech', 'offensive', or 'normal'."
    )

    user_prompt = f"Text: {text}"

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0, # Deterministic
            max_tokens=10,
            timeout=60
        )
        return response.choices[0].message.content.strip().lower()
    except Exception as e:
        print(f"Error: {e}")
        return "error"

sample_text = test_ds[0]['text_norm']
sample_true_label = ID2NAME[test_ds[0]['label_norm']]
print(f"Sample Text: {sample_text}")
print(f"Ground Truth: {sample_true_label}")
print(f"Prediction: {classify_text(sample_text)}")


### Prompt 2

# @title Define Prompt & Run on 1 Example


def classify_text(text, model="gemma3:4b"):
    """
    Classifies text using Local Ollama API.
    Make sure you have pulled the model: `ollama pull phi3`
    """
    system_prompt = (
"You are a strict three-way classifier: hatespeech, offensive, normal.\n"
"Return ONLY one label in lowercase.\n\n"

"HARD GATE 1 (Hatespeech needs both):"
"- Protected-group reference must be explicit AND"
"- Identity-attack signal must be explicit."
"If either is missing -> NOT hatespeech."
"Protected-group reference must be a specific protected identity label (e.g., Muslims, Jews, Black people, gay people, women)."
"Generic collectives like 'immigrants', 'foreigners', 'people', 'they', 'the left/right', 'men' (unless explicitly gender-targeted), 'kids', 'boomers' do NOT qualify by themselves."
"Identity-attack signal must be one of:"
"- slur"
"- dehumanization (vermin/animals/disease/filth)"
"- group inferiority (stupid/inferior by nature)"
"- violence/extermination"
"- forced removal/exclusion framed as hostility ('kick them out', 'ban them' AND accompanied by contempt/hate framing (not policy debate)."
"If it’s just policy preference without contempt/dehumanization/slurs -> NOT hatespeech."

"HARD GATE 2 (Offensive requires a clear offense marker):"
"Offensive ONLY if there is explicit profanity/vulgarity OR a direct insult/name-calling/harassment."
"If none -> normal."

"STEP-BY-STEP DECISION PROCEDURE:\n"
"1) QUOTES / CONTEXT CHECK:\n"
"   If the text is quoting hate/offense to condemn, report, educate, or discuss neutrally -> normal.\n"
"2) HATESPEECH CHECK (must satisfy A + B):\n"
"   A) Target is a PROTECTED GROUP (race, ethnicity, nationality, religion, disability, age,\n"
"      sexual orientation, gender, gender identity, veteran status).\n"
"   B) The text attacks the group on identity via at least one:\n"
"      - dehumanization (animals/filth/disease)\n"
"      - calls for violence/extermination/forced removal\n"
"      - explicit group inferiority / group-wide hatred\n"
"      - slurs used to degrade the group\n"
"   If A+B true -> hatespeech.\n"
"3) OFFENSIVE CHECK (must contain a clear offense signal):\n"
"   Label offensive ONLY if there is at least one of:\n"
"   - directed insult/name-calling toward a person or non-protected group (e.g., idiot, dumbass)\n"
"   - profanity used aggressively (not just mild emphasis) OR sexual/scatological vulgarity\n"
"   - harassment/demeaning commands aimed at someone (shut up, kill yourself, etc.)\n"
"   If none of these are present -> do NOT choose offensive.\n"
"4) NORMAL:\n"
"   Everything else is normal, including:\n"
"   - disagreement, criticism, sarcasm without insults/profanity\n"
"   - criticism of ideas, policies, religions, ideologies, governments (without attacking believers as a group)\n"
"   - neutral/factual discussion of sensitive topics\n"
"   - product text, listings, random tokens, casual chat\n\n"

"BOUNDARY CLARIFICATIONS:\n"
"- Criticizing a religion/ideology is normal unless it generalizes believers as inferior/filthy/violent or calls harm.\n"
"- Mentioning a protected group alone is normal.\n"
"- “Group X hates Y” is NOT automatically hatespeech; it becomes hatespeech only if it asserts identity-based inferiority,\n"
"  dehumanizes, uses slurs, or advocates harm/exclusion.\n\n"

"When unsure between normal vs (offensive/hatespeech), choose normal."

"Don't be too sensitive and classify everything as offensive."

"OUTPUT FORMAT:\n"
"Return ONLY one word: hatespeech OR offensive OR normal."
)

    user_prompt = f"Text: {text}"

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0, # Deterministic
            max_tokens=10,
            timeout=60
        )
        return response.choices[0].message.content.strip().lower()
    except Exception as e:
        print(f"Error: {e}")
        return "error"

print(f"Full Entry: {test_ds[1]}")
sample_text, sample_true_label = test_ds[42]['text_norm'], ID2NAME[test_ds[42]['label_norm']] # change both the indexes to match
# label norm = 0 hatespeech , 1 normal , 2 offensive
print(f"Sample Text: {sample_text}")
print(f"Ground Truth: {sample_true_label}")
print(f"Prediction: {classify_text(sample_text)}")